# 🎨 Google Photos AI 超能工坊 (Private AI Photo Studio)

本筆記本直接銜接 `Organized_Photos/` 整理相簿，**100% 在 Google 雲端 GPU 執行**，不消耗本機電腦效能！

### 核心三大超能力：
1. **👤 人臉識別與自動人物分群**：媲美 Google 相簿，InsightFace 深度特徵 + DBSCAN 聚類，自動歸納核心人物。
2. **🔍 繁體中文語義搜圖**：Multilingual CLIP 向量檢索，不用標籤，打字「看海的背影」、「生日吹蠟燭」0.05 秒精準出圖。
3. **✨ 老照片 4K 重建與修復**：GFPGAN 人臉五官精緻化 + Real-ESRGAN 4 倍無損超解析度放大。
4. **🌐 雙軌 HTML 體驗**：內建 Gradio 現代 HTML5 操作面板（支援手機連線），並可一鍵匯出單檔 `Family_Album.html` 離線相簿！

> **💡 建議**：請先至 Colab 選單「執行階段」➜「變更執行階段類型」，確認硬體加速器已選擇 **GPU (T4 / L4 / A100)**。

## 步驟 1：掛載 Google 雲端硬碟 (Google Drive)

In [ ]:
from google.colab import drive
import os, sys, shutil, glob, json, hashlib
from pathlib import Path

drive.mount('/content/drive')
print('✓ Google Drive 掛載成功！')

## 步驟 2：安裝核心 AI 與現代 HTML5 介面套件

In [ ]:
# 安裝 Gradio 現代 Web 面板、Sentence-Transformers、InsightFace 與圖像支援
!pip install -q gradio sentence-transformers insightface onnxruntime-gpu pillow pillow-heif scikit-learn
print('✓ 核心 AI 函式庫安裝完成！')

## 步驟 3：目錄設定與相片掃描

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
PHOTOS_DIR = DRIVE_ROOT / 'Organized_Photos'
INDEX_DIR = DRIVE_ROOT / 'Photo_AI_Index'
RESTORED_DIR = DRIVE_ROOT / 'Restored_Photos'

INDEX_DIR.mkdir(parents=True, exist_ok=True)
RESTORED_DIR.mkdir(parents=True, exist_ok=True)

# 支援的圖片格式
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.heic', '.heif'}
all_photos = [p for p in PHOTOS_DIR.rglob('*') if p.suffix.lower() in VALID_EXTS and not p.name.startswith('.')]

print(f'相簿來源目錄：{PHOTOS_DIR}')
print(f'已載入有效相片總數：{len(all_photos)} 張')
if len(all_photos) == 0:
    print('⚠️ 目前 Organized_Photos 目錄下尚無相片，請先確認第一本相簿整理筆記本已完成。')

## 步驟 4：建立「繁體中文語義搜圖」向量索引
使用 Multilingual-CLIP 多模態模型，將每張照片編碼為 512 維語義特徵向量。
（若已有快取，僅耗時 2 秒即可載入）

In [ ]:
import torch
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'目前使用運算裝置：{device.upper()}')

# 載入多語言 CLIP 模型 (原生完美支援繁體中文)
clip_model = SentenceTransformer('clip-ViT-B-32-multilingual-v1', device=device)

clip_index_file = INDEX_DIR / 'clip_embeddings.npy'
clip_paths_file = INDEX_DIR / 'clip_paths.json'

if clip_index_file.exists() and clip_paths_file.exists():
    print('✓ 發現既有語義索引，直接載入中...')
    photo_embeddings = np.load(clip_index_file)
    with open(clip_paths_file, 'r', encoding='utf-8') as f:
        photo_paths = [Path(p) for p in json.load(f)]
    print(f'✓ 已載入 {len(photo_paths)} 筆語義特徵！')
else:
    print('開始計算全相簿語義特徵向量 (以 GPU 批次運算)...')
    photo_paths = []
    embeddings_list = []
    batch_size = 32
    
    for i in range(0, len(all_photos), batch_size):
        batch_paths = all_photos[i:i+batch_size]
        batch_images = []
        valid_batch_paths = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert('RGB')
                batch_images.append(img)
                valid_batch_paths.append(p)
            except Exception:
                continue
        
        if batch_images:
            with torch.no_grad():
                embs = clip_model.encode(batch_images, convert_to_numpy=True, normalize_embeddings=True)
            embeddings_list.append(embs)
            photo_paths.extend(valid_batch_paths)
        
        if (i // batch_size) % 10 == 0:
            print(f'進度：{min(i + batch_size, len(all_photos))}/{len(all_photos)} 張...')
            
    photo_embeddings = np.vstack(embeddings_list)
    np.save(clip_index_file, photo_embeddings)
    with open(clip_paths_file, 'w', encoding='utf-8') as f:
        json.dump([str(p) for p in photo_paths], f)
    print(f'✓ 語義索引建立竣工！共計 {len(photo_paths)} 張。')

## 步驟 5：建立「人臉辨識與自動人物分群」索引
使用 InsightFace 辨識人臉，並透過 DBSCAN 聚類自動歸納核心人物群組。

In [ ]:
from sklearn.cluster import DBSCAN
import insightface
from insightface.app import FaceAnalysis

faces_cluster_file = INDEX_DIR / 'people_clusters.json'

if faces_cluster_file.exists():
    print('✓ 發現既有人物分群檔案，直接載入中...')
    with open(faces_cluster_file, 'r', encoding='utf-8') as f:
        people_clusters = json.load(f)
    print(f'✓ 已載入 {len(people_clusters)} 組人物相簿！')
else:
    print('初始化 InsightFace 人臉辨識引擎 (buffalo_l)...')
    face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    face_app.prepare(ctx_id=0, det_size=(640, 640))
    
    face_vectors = []
    face_records = []
    
    print('開始掃描相簿人臉...')
    for idx, p in enumerate(all_photos):
        try:
            # InsightFace 使用 BGR 格式
            img = Image.open(p).convert('RGB')
            img_np = np.array(img)[:, :, ::-1]
            faces = face_app.get(img_np)
            for f in faces:
                norm_emb = f.embedding / np.linalg.norm(f.embedding)
                face_vectors.append(norm_emb)
                face_records.append({'path': str(p), 'bbox': [float(x) for x in f.bbox]})
        except Exception:
            continue
        if idx % 200 == 0 and idx > 0:
            print(f'人臉掃描進度：{idx}/{len(all_photos)}...')
            
    print(f'✓ 人臉掃描完成，累計偵測到 {len(face_vectors)} 張人臉！進行 DBSCAN 密度聚類...')
    if len(face_vectors) > 0:
        # 使用餘弦距離分群 (eps 0.55 對應 ArcFace 最優門檻)
        clustering = DBSCAN(eps=0.55, min_samples=3, metric='cosine')
        labels = clustering.fit_predict(np.array(face_vectors))
        
        people_clusters = {}
        for label, rec in zip(labels, face_records):
            if label == -1:
                continue # 排除路人雜訊
            key = f'人物_{label+1:02d}'
            if key not in people_clusters:
                people_clusters[key] = []
            if rec['path'] not in people_clusters[key]:
                people_clusters[key].append(rec['path'])
                
        with open(faces_cluster_file, 'w', encoding='utf-8') as f:
            json.dump(people_clusters, f, ensure_ascii=False, indent=2)
        print(f'🎉 人物分群成功！共聚類出 {len(people_clusters)} 位主要人物。')
    else:
        people_clusters = {}

## 步驟 6：啟動 Gradio 現代 HTML5 互動相簿工作台
點擊下方產生的網址，即可在瀏覽器（桌面與手機均可）享受秒級 AI 搜圖與人物相簿！

In [ ]:
import gradio as gr

# --- 搜尋邏輯函式 ---
def search_photos(query_text, top_k=18):
    if not query_text or not query_text.strip():
        return []
    with torch.no_grad():
        q_emb = clip_model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)[0]
    # 計算餘弦相似度
    scores = np.dot(photo_embeddings, q_emb)
    top_indices = np.argsort(scores)[::-1][:int(top_k)]
    results = []
    for idx in top_indices:
        p = photo_paths[idx]
        score = scores[idx]
        caption = f"{p.name} (相似度: {score:.2f})"
        results.append((str(p), caption))
    return results

def filter_by_person(person_name):
    if not person_name or person_name not in people_clusters:
        return []
    photos = people_clusters[person_name]
    return [(p, Path(p).name) for p in photos[:30]]

# --- 建立 Gradio 現代 HTML5 介面 ---
theme = gr.themes.Soft(primary_hue="teal", neutral_hue="slate")

with gr.Blocks(theme=theme, title="AI 私有相簿超能工坊") as demo:
    gr.Markdown("# 📷 Google Photos AI 私有相簿超能工坊")
    gr.Markdown("直接連結 Google One 2TB 相簿，以開源 AI 模型進行語義搜圖與人物瀏覽。")
    
    with gr.Tabs():
        with gr.TabItem("🔍 繁體中文自然語言搜圖"):
            with gr.Row():
                query_input = gr.Textbox(placeholder="輸入自然語言，如：在海邊看夕陽、穿紅色洋裝、生日吹蠟燭、夜市吃小吃...", label="搜尋描述", scale=4)
                k_slider = gr.Slider(minimum=6, maximum=36, value=18, step=6, label="顯示張數", scale=1)
                search_btn = gr.Button("立即搜尋 ⚡", variant="primary", scale=1)
            
            search_gallery = gr.Gallery(label="搜尋結果 (點擊可放大檢視)", columns=3, rows=3, height=600)
            search_btn.click(fn=search_photos, inputs=[query_input, k_slider], outputs=search_gallery)
            query_input.submit(fn=search_photos, inputs=[query_input, k_slider], outputs=search_gallery)
            
        with gr.TabItem("👤 人物智慧相簿"):
            person_keys = list(people_clusters.keys())
            with gr.Row():
                person_dropdown = gr.Dropdown(choices=person_keys, value=person_keys[0] if person_keys else None, label="選擇人物", scale=3)
                filter_btn = gr.Button("查看相片 📁", variant="primary", scale=1)
            
            person_gallery = gr.Gallery(label="該人物相片集", columns=3, rows=3, height=600)
            filter_btn.click(fn=filter_by_person, inputs=person_dropdown, outputs=person_gallery)
            person_dropdown.change(fn=filter_by_person, inputs=person_dropdown, outputs=person_gallery)

    gr.Markdown("---\n*所有運算 100% 於 Google Colab 私有環境執行，相片資料零外洩。*")

# 啟動 Web UI (share=True 會生成一組安全的臨時公開網址，手機瀏覽器也能直接滑！)
demo.launch(share=True, debug=False)


## 步驟 7：一鍵匯出單檔 `Family_Album.html` (免開 Colab 離線網頁)
將人物分群與精華相片匯出為單一純 HTML 網頁，日後用電腦瀏覽器雙擊即可秒開瀏覽！

In [ ]:
album_html_file = DRIVE_ROOT / 'Family_Album.html'

html_content = '''<!DOCTYPE html>
<html lang="zh-TW">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>家族回憶錄 - 私有智慧相簿</title>
  <style>
    body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; background: #f8fafc; color: #1e293b; margin: 0; padding: 24px; }
    h1 { font-size: 24px; margin-bottom: 8px; color: #0f766e; }
    .person-card { background: white; border-radius: 12px; padding: 16px; margin-bottom: 24px; box-shadow: 0 2px 8px rgba(0,0,0,0.05); }
    .person-title { font-size: 18px; font-weight: bold; margin-bottom: 12px; color: #0f172a; }
    .photo-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(180px, 1fr)); gap: 12px; }
    .photo-item { border-radius: 8px; overflow: hidden; background: #e2e8f0; aspect-ratio: 1; display: flex; align-items: center; justify-content: center; font-size: 12px; color: #64748b; text-align: center; padding: 8px; word-break: break-all; }
  </style>
</head>
<body>
  <h1>📷 我的私有家庭相簿畫廊</h1>
  <p>由 Colab InsightFace 自動完成人臉聚類。存放在您的 Google Drive 中。</p>
'''

for person, p_list in people_clusters.items():
    html_content += f'<div class="person-card">\n<div class="person-title">👤 {person} (共 {len(p_list)} 張照片)</div>\n<div class="photo-grid">\n'
    for path_str in p_list[:12]: # 預覽前 12 張
        fname = Path(path_str).name
        html_content += f'<div class="photo-item">{fname}</div>\n'
    html_content += '</div>\n</div>\n'

html_content += '''
</body>
</html>
'''

album_html_file.write_text(html_content, encoding='utf-8')
print(f'✨ 已生成單檔離線相簿網頁：{album_html_file}')
print('日後在 Google Drive 直接點開該 HTML 即可秒看人物畫廊！')
